# Session 6: HuggingFace & Transformers (45 minutes)

## 🎯 Learning Objectives
- Understand the HuggingFace ecosystem
- Use open-source models locally
- Integrate HuggingFace with LangChain
- Run inference with Transformers

## 📋 Problem Statement
Sometimes you need:
- Privacy: Keep data on-premise
- Cost: Avoid API costs
- Control: Fine-tune for your use case
- Latency: Run locally for speed

## ⏱️ Session Breakdown
- 5 min: HuggingFace overview
- 15 min: Transformers library basics
- 15 min: HuggingFace + LangChain integration
- 5 min: HuggingFace Hub models
- 5 min: Recap

---

## 1. Setup & Dependencies

In [ ]:
# Install HuggingFace packages if needed
# !pip install transformers torch accelerate sentence-transformers
# !pip install langchain-huggingface

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_ollama import ChatOllama

# Option 1: Local via Ollama (FREE, NO API!)
ollama_llm = ChatOllama(model="qwen2:0.5b", temperature=0.7)

print("✅ Session 6 Setup Complete!")
print("🖥️  Using TinyLlama via Ollama - LOCAL, FREE, NO LIMITS!")

## 2. The HuggingFace Ecosystem

🤗 **HuggingFace** is the go-to platform for open-source ML:

```
┌─────────────────────────────────────────────────────────────┐
│                  HUGGINGFACE ECOSYSTEM                       │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  🏠 HuggingFace Hub          🔧 Transformers Library         │
│  ├── 500K+ models            ├── Load & run models          │
│  ├── Datasets                ├── Tokenization               │
│  ├── Spaces (demos)          ├── Inference pipelines        │
│  └── Model cards             └── Training utilities         │
│                                                              │
│  📊 sentence-transformers    🔗 LangChain Integration        │
│  └── Embeddings              └── langchain-huggingface      │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

## 3. Transformers Library Basics

In [ ]:
from transformers import pipeline

# The easiest way to use models - pipelines!
print("📦 Pipeline: High-level API for common tasks")
print("""  
Available pipelines:
- 'text-generation': Generate text
- 'sentiment-analysis': Classify sentiment
- 'question-answering': Answer questions from context
- 'summarization': Summarize text
- 'translation': Translate between languages
- 'fill-mask': Fill in blanked words
- And many more!
""")

## 4. Sentiment Analysis Pipeline

In [ ]:
# Create a sentiment analysis pipeline
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Test sentences
sentences = [
    "I love this workshop! It's amazing!",
    "This is confusing and frustrating.",
    "The weather is okay today."
]

print("😊 Sentiment Analysis Demo\n" + "="*50)
for sentence in sentences:
    result = sentiment_pipeline(sentence)[0]
    emoji = "😊" if result['label'] == 'POSITIVE' else "😔"
    print(f"{emoji} {result['label']}: {result['score']:.3f}")
    print(f"   Text: {sentence}\n")

## 5. Text Summarization Pipeline

In [ ]:
# Create a summarization pipeline
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

# Sample text to summarize
long_text = """
Artificial intelligence (AI) has transformed numerous industries over the past decade. 
Machine learning, a subset of AI, enables computers to learn from data without being 
explicitly programmed. Deep learning, which uses neural networks with many layers, 
has achieved remarkable results in image recognition, natural language processing, 
and speech recognition. Companies are now using AI for everything from customer 
service chatbots to autonomous vehicles. However, there are also concerns about 
AI safety, job displacement, and the ethical implications of these powerful systems. 
Researchers and policymakers are working to ensure AI develops in a beneficial 
direction for humanity.
"""

print("📝 Text Summarization Demo\n" + "="*50)
print(f"Original ({len(long_text.split())} words):")
print(f"{long_text[:200]}...\n")

summary = summarizer(long_text, max_length=50, min_length=20, do_sample=False)
print(f"Summary:")
print(summary[0]['summary_text'])

## 6. Question Answering Pipeline

In [ ]:
# Create a QA pipeline
qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad"
)

# Context and questions
context = """
LangChain is a framework for developing applications powered by language models.
It was created by Harrison Chase and released in October 2022. LangChain provides
tools for building chains, agents, and retrieval-augmented generation systems.
The framework supports multiple language model providers including OpenAI, 
Anthropic, Google, and HuggingFace.
"""

questions = [
    "Who created LangChain?",
    "When was LangChain released?",
    "What providers does LangChain support?"
]

print("❓ Question Answering Demo\n" + "="*50)
print(f"Context: {context[:100]}...\n")

for question in questions:
    result = qa_pipeline(question=question, context=context)
    print(f"Q: {question}")
    print(f"A: {result['answer']} (confidence: {result['score']:.3f})\n")

## 7. Text Generation (Caution: Resource Intensive)

In [ ]:
# Note: Large language models require significant resources
# For this demo, we use a small model

# Create text generation pipeline with a small model
generator = pipeline(
    "text-generation",
    model="gpt2",  # Small model (124M params)
    device_map="auto"  # Use GPU if available
)

prompt = "The future of artificial intelligence is"

print("✍️ Text Generation Demo\n" + "="*50)
print(f"Prompt: {prompt}\n")

result = generator(
    prompt,
    max_new_tokens=50,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.7
)

print(f"Generated: {result[0]['generated_text']}")

## 8. HuggingFace Embeddings in LangChain

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# Create embeddings using a free, local model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Test the embeddings
texts = [
    "Machine learning is fascinating",
    "I love neural networks",
    "The weather is nice today"
]

print("🔢 HuggingFace Embeddings Demo\n" + "="*50)

# Embed documents
doc_embeddings = embeddings.embed_documents(texts)

for text, emb in zip(texts, doc_embeddings):
    print(f"Text: {text}")
    print(f"  Dimension: {len(emb)}")
    print(f"  First 5 values: {emb[:5]}\n")

## 9. Semantic Similarity with HuggingFace Embeddings

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Compare semantic similarity
sentences = [
    "I love programming",
    "Coding is my passion",
    "I enjoy cooking"
]

reference = "Software development is fun"
ref_embedding = embeddings.embed_query(reference)

print("🔍 Semantic Similarity Demo\n" + "="*50)
print(f"Reference: '{reference}'\n")

for sentence in sentences:
    sent_embedding = embeddings.embed_query(sentence)
    similarity = cosine_similarity(ref_embedding, sent_embedding)
    print(f"  '{sentence}': {similarity:.4f}")

## 10. HuggingFace LLMs in LangChain (Pipeline)

In [ ]:
from langchain_huggingface import HuggingFacePipeline

# Create a LangChain LLM from a HuggingFace pipeline
hf_llm = HuggingFacePipeline.from_model_id(
    model_id="gpt2",
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 100,
        "temperature": 0.7
    }
)

print("✅ HuggingFace Pipeline LLM created!")

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Create a chain with HuggingFace LLM
prompt = PromptTemplate.from_template(
    "Write a short poem about {topic}:\n\n"
)

hf_chain = prompt | hf_llm | StrOutputParser()

print("📝 HuggingFace + LangChain Demo\n" + "="*50)
result = hf_chain.invoke({"topic": "artificial intelligence"})
print(result)

## 11. Using HuggingFace Hub Models (API)

In [ ]:
# For HuggingFace Inference API (requires HF token)
# Set HUGGINGFACEHUB_API_TOKEN in your .env file

from langchain_huggingface import HuggingFaceEndpoint

# Check if HF token is available
hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

if hf_token:
    # Use HuggingFace Inference API
    hf_endpoint = HuggingFaceEndpoint(
        repo_id="mistralai/Mistral-7B-Instruct-v0.2",
        max_new_tokens=200,
        temperature=0.7
    )
    print("✅ HuggingFace Endpoint created!")
else:
    print("⚠️ No HUGGINGFACEHUB_API_TOKEN found")
    print("To use HF Inference API:")
    print("  1. Get a token from https://huggingface.co/settings/tokens")
    print("  2. Add HUGGINGFACEHUB_API_TOKEN=your-token to .env")

## 12. Custom Model Loading

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Manual model loading gives more control
print("📥 Manual Model Loading Demo\n" + "="*50)

model_id = "gpt2"

# Load tokenizer and model separately
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,  # Use float16 for GPU
    device_map="auto"
)

print(f"✅ Loaded model: {model_id}")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Manual inference
input_text = "The key to success in AI is"
inputs = tokenizer(input_text, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\nInput: {input_text}")
print(f"Output: {generated}")

## 13. Specialized Models for Different Tasks

In [ ]:
print("""🤗 Popular HuggingFace Models by Task
════════════════════════════════════════════════════════════════

📝 TEXT GENERATION (Chat/Completion):
   - mistralai/Mistral-7B-Instruct-v0.2    (7B, great balance)
   - meta-llama/Llama-2-7b-chat-hf         (7B, Meta's model)
   - tiiuae/falcon-7b-instruct             (7B, fast)
   - google/gemma-2b-it                     (2B, lightweight)

🔢 EMBEDDINGS:
   - sentence-transformers/all-MiniLM-L6-v2    (Fast, 384 dim)
   - sentence-transformers/all-mpnet-base-v2  (Better quality)
   - BAAI/bge-small-en-v1.5                   (State-of-art)

😊 SENTIMENT/CLASSIFICATION:
   - distilbert-base-uncased-finetuned-sst-2-english
   - cardiffnlp/twitter-roberta-base-sentiment

❓ QUESTION ANSWERING:
   - distilbert-base-cased-distilled-squad
   - deepset/roberta-base-squad2

📋 SUMMARIZATION:
   - facebook/bart-large-cnn
   - google/pegasus-xsum

════════════════════════════════════════════════════════════════
""")

## 14. Using HuggingFace for RAG

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# Create sample documents
docs = [
    Document(page_content="LangChain is a framework for building LLM applications."),
    Document(page_content="HuggingFace provides open-source machine learning models."),
    Document(page_content="RAG combines retrieval with generation for better answers."),
    Document(page_content="Transformers are the architecture behind modern LLMs.")
]

# Create vector store with HuggingFace embeddings
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="hf_demo"
)

# Test retrieval
query = "What is LangChain used for?"
results = vectorstore.similarity_search(query, k=2)

print("🔍 RAG with HuggingFace Embeddings\n" + "="*50)
print(f"Query: {query}\n")
print("Retrieved Documents:")
for i, doc in enumerate(results, 1):
    print(f"  {i}. {doc.page_content}")

## 15. Comparing Cloud vs Local Models

In [ ]:
prompt = "Explain what machine learning is in one sentence."

print("⚖️ Cloud vs Local Model Comparison\n" + "="*50)

# Cloud model (Gemini)
print("\n☁️ Cloud (Gemma 3 27B via Gemini API):")
cloud_response = gemini_llm.invoke(prompt)
print(f"  {cloud_response.content}")

# Local model (GPT-2)
print("\n💻 Local (GPT-2):")
local_response = hf_llm.invoke(prompt)
print(f"  {local_response[:200]}...")

print("\n📊 Comparison:")
print("""
┌─────────────────┬─────────────────┬─────────────────┐
│    Aspect       │     Cloud       │     Local       │
├─────────────────┼─────────────────┼─────────────────┤
│ Quality         │ Higher          │ Lower           │
│ Speed           │ Network latency │ GPU dependent   │
│ Cost            │ Per token       │ Free (compute)  │
│ Privacy         │ Data sent out   │ Data stays local│
│ Customization   │ Limited         │ Full control    │
└─────────────────┴─────────────────┴─────────────────┘
""")

## 📚 Session 6 Recap

### Key Takeaways:

1. **HuggingFace Ecosystem:**
   - Hub: 500K+ models
   - Transformers: Load & run models
   - sentence-transformers: Embeddings

2. **Pipelines:**
   - Easy high-level API
   - Tasks: generation, QA, summarization, sentiment

3. **LangChain Integration:**
   - `HuggingFaceEmbeddings` for RAG
   - `HuggingFacePipeline` for local LLMs
   - `HuggingFaceEndpoint` for Inference API

4. **When to Use Local Models:**
   - Privacy requirements
   - Cost optimization
   - Custom fine-tuning needs
   - Offline operation

---

### 🔜 Next Session: Complete Agent
"Let's put everything together into a production-ready Research Assistant!"

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════════════╗
║                    SESSION 6 COMPLETE! 🎉                                  ║
║                                                                            ║
║  ☕ SHORT BREAK - 5 MINUTES ☕                                              ║
║                                                                            ║
║  Next: Session 7 - Complete Research Agent                                 ║
║  File: 07_complete_agent.ipynb                                             ║
║                                                                            ║
║  "Time to build our full Research Assistant!"                              ║
║  "Combining everything: RAG, Agents, Tools, Memory"                        ║
╚═══════════════════════════════════════════════════════════════════════════╝
""")